In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

In [43]:
columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in","num_compromised",
    "root_shell","su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login","is_guest_login","count",
    "srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate",
    "same_srv_rate","diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label","difficulty"
]

In [44]:
df_train = pd.read_csv("../data/raw/KDDTrain+.txt", header=None, names=columns)
df_test = pd.read_csv("../data/raw/KDDTest+.txt", header=None, names=columns)
print(df_train.shape, df_test.shape)
datas = pd.DataFrame(df_train)
datas.head(5)

(125973, 43) (22544, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [45]:
x_train = df_train.drop(["label", "difficulty"], axis = 1)
y_train = df_train["label"]
y_train_binary = (y_train != "normal").astype(int)
x_test = df_test.drop(["label", "difficulty"], axis = 1)
y_test = df_test["label"]
y_test_binary = (y_test != "normal").astype(int)

In [46]:
x_train_encoded = pd.get_dummies(x_train)
x_test_encoded = pd.get_dummies(x_test)
x_test_encoded = x_test_encoded.reindex(columns=x_train_encoded.columns, fill_value=0)
print(x_train_encoded.shape)
print(x_test_encoded.shape)


(125973, 122)
(22544, 122)


In [47]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_encoded)
x_test_scaled = scaler.transform(x_test_encoded)
print(x_train_scaled.shape)
print(x_test_scaled.shape)

(125973, 122)
(22544, 122)


In [48]:
np.random.seed(42)
label_fraction = 0.05
n_samples = x_train_scaled.shape[0]
n_labeled = int(n_samples * label_fraction)
labeled_indices = np.random.choice(n_samples, size=n_labeled, replace=False)
is_labeled = np.zeros(n_samples, dtype=bool)
is_labeled[labeled_indices] = True

print(f"Total samples: {n_samples}")
print(f"Labeled samples: {is_labeled.sum()} ({is_labeled.sum()/n_samples:.1%})")
print(f"Unlabeled samples: {(~is_labeled).sum()} ({(~is_labeled).sum()/n_samples:.1%})")

Total samples: 125973
Labeled samples: 6298 (5.0%)
Unlabeled samples: 119675 (95.0%)


In [49]:
x_labeled = x_train_scaled[is_labeled]
y_labeled = y_train_binary[is_labeled]
y_unlabeled_true = y_train_binary[~is_labeled]
x_unlabeled = x_train_scaled[~is_labeled]

print(x_labeled.shape, y_labeled.shape)
print(x_unlabeled.shape, y_unlabeled_true.shape)

(6298, 122) (6298,)
(119675, 122) (119675,)


In [50]:
clf = LogisticRegression(max_iter = 1000, random_state=42)
clf.fit(x_labeled, y_labeled)

probs = clf.predict_proba(x_unlabeled)
attack_prob = probs[:, 1]
print(attack_prob[:10])

[0.00970282 0.06959076 0.99999959 0.00389487 0.01691357 0.99928081
 0.99999982 0.99999952 0.99999979 0.99999969]


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/line

In [51]:
high_threshold = .95
low_threshold = .05 
confident_attack = attack_prob >= high_threshold
confident_normal = attack_prob <= low_threshold
confident_mask = confident_attack | confident_normal

print(f"Confident predictions: {confident_mask.sum()} out of {len(attack_prob)} ({confident_mask.sum()/len(attack_prob):.1%})")
print(f"  - confident attack: {confident_attack.sum()}")
print(f"  - confident normal: {confident_normal.sum()}")

Confident predictions: 106816 out of 119675 (89.3%)
  - confident attack: 49829
  - confident normal: 56987


In [52]:
pseudo_labeles = np.where(confident_attack, 1,0)
pseudo_labels_confident = pseudo_labeles[confident_mask]
true_labels_confident = y_unlabeled_true[confident_mask]

accuracy = (pseudo_labels_confident == true_labels_confident).mean()
print(f"Pseudo-label accuracy on confident subset: {accuracy:.4f}")


Pseudo-label accuracy on confident subset: 0.9937


In [53]:
x_combined = np.vstack([x_labeled, x_unlabeled[confident_mask]])
y_combined = np.concatenate([y_labeled, pseudo_labels_confident])
print(x_combined.shape, y_combined.shape)
clf_self_trained = LogisticRegression(max_iter=1000, random_state=42)
clf_self_trained.fit(x_combined, y_combined)

y_pred_test = clf_self_trained.predict(x_test_scaled)

print(confusion_matrix(y_test_binary, y_pred_test))
print("Precision:", precision_score(y_test_binary, y_pred_test))
print("Recall:", recall_score(y_test_binary, y_pred_test))
print("F1 Score:", f1_score(y_test_binary, y_pred_test))





(113114, 122) (113114,)


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/line

[[8983  728]
 [4945 7888]]
Precision: 0.9155060352831941
Recall: 0.6146653159822333
F1 Score: 0.7355121450883491


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [54]:
clf_baseline = LogisticRegression(max_iter = 1000, random_state=42)
clf_baseline.fit(x_labeled, y_labeled)
y_pred_baseline = clf_baseline.predict(x_test_scaled)
print(confusion_matrix(y_test_binary, y_pred_baseline))
print("Precision:", precision_score(y_test_binary, y_pred_baseline))
print("Recall:", recall_score(y_test_binary, y_pred_baseline))
print("F1 Score:", f1_score(y_test_binary, y_pred_baseline))


[[8995  716]
 [4897 7936]]
Precision: 0.9172445677300046
Recall: 0.6184056728746201
F1 Score: 0.7387479636956016


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/sklearn/line

In [55]:
x_normal = x_train_scaled[(y_train_binary == 0).to_numpy()]
print(x_normal.shape)
mu = x_normal.mean(axis =0)
cov = np.cov(x_normal, rowvar= False)
print(mu.shape, cov.shape)
print("rank of covariance:", np.linalg.matrix_rank(cov))

(67343, 122)
(122,) (122, 122)
rank of covariance: 73


In [56]:
cov_inv = np.linalg.pinv(cov)
def mahalanobis_sq(X, mu,cov_inv):
    diff = X - mu 
    return ((diff @ cov_inv) * diff).sum(axis=1)   # (x - mu)^T * cov_inv * (x - mu), per row

d2_test = mahalanobis_sq(x_test_scaled, mu, cov_inv)
print(d2_test.shape)
print(d2_test[:5])

(22544,)
[593.6488987  500.99140121  39.63886748 188.55684905 665.98077178]


/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/numpy/linalg/_linalg.py:3220: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/numpy/linalg/_linalg.py:3220: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/sayoowo5/CS4360-Senior-Experience-Project/.venv/lib/python3.9/site-packages/numpy/linalg/_linalg.py:3220: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)
/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_11427/3576951426.py:4: RuntimeWarning: divide by zero encountered in matmul
  return ((diff @ cov_inv) * diff).sum(axis=1)   # (x - mu)^T * cov_inv * (x - mu), per row
/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_11427/3576951426.py:4: RuntimeWarning: overflow encountered in matmul
  return ((diff @ cov_inv) * diff).sum(axis=1)   # (x - mu)^T *

In [57]:
d2_train_normal = mahalanobis_sq(x_normal, mu, cov_inv)
threshold = np.percentile(d2_train_normal, 95)
print("threshold:", threshold)
y_pred_maha = (d2_test > threshold).astype(int)
print(confusion_matrix(y_test_binary, y_pred_maha))
print("Precision:", precision_score(y_test_binary, y_pred_maha))
print("Recall:", recall_score(y_test_binary, y_pred_maha))
print("F1 Score:", f1_score(y_test_binary, y_pred_maha))

threshold: 217.0569593534439
[[9010  701]
 [3514 9319]]
Precision: 0.9300399201596806
Recall: 0.7261747058365152
F1 Score: 0.8155603203080558


/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_11427/3576951426.py:4: RuntimeWarning: divide by zero encountered in matmul
  return ((diff @ cov_inv) * diff).sum(axis=1)   # (x - mu)^T * cov_inv * (x - mu), per row
/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_11427/3576951426.py:4: RuntimeWarning: overflow encountered in matmul
  return ((diff @ cov_inv) * diff).sum(axis=1)   # (x - mu)^T * cov_inv * (x - mu), per row
/var/folders/fd/bwkz9x5n2glf3hs_rv6k29zc0000gn/T/ipykernel_11427/3576951426.py:4: RuntimeWarning: invalid value encountered in matmul
  return ((diff @ cov_inv) * diff).sum(axis=1)   # (x - mu)^T * cov_inv * (x - mu), per row


In [58]:
for pct in [80, 90, 95, 99]:
    t = np.percentile(d2_train_normal, pct)
    pred = (d2_test > t).astype(int)
    print(f"{pct}th pct | threshold {t:8.1f} | precision {precision_score(y_test_binary, pred):.4f} | "
          f"recall {recall_score(y_test_binary, pred):.4f} | f1 {f1_score(y_test_binary, pred):.4f}")

80th pct | threshold     49.8 | precision 0.8773 | recall 0.8534 | f1 0.8652
90th pct | threshold    130.5 | precision 0.9103 | recall 0.7712 | f1 0.8350
95th pct | threshold    217.1 | precision 0.9300 | recall 0.7262 | f1 0.8156
99th pct | threshold    914.5 | precision 0.9753 | recall 0.2583 | f1 0.4085
